In [2]:
# verification before starting the training 
#1 Check config.cfg has correct settings
with open('../config.cfg', 'r') as f:
    config = f.read()

# Check these 3 critical lines
checks = {
    'vectors = null':        'vectors set to null',
    'train.spacy':           'train path present',
    'dev.spacy':             'dev path present',
}

print("Config checks:")
for key, label in checks.items():
    found = key in config
    print(f"  {'✅' if found else '❌'} {label}")

# Print relevant lines
print("\nRelevant config lines:")
for line in config.split('\n'):
    if any(x in line for x in ['vectors','train','dev','gpu','batch']):
        print(f"  {line.strip()}")

Config checks:
  ✅ vectors set to null
  ✅ train path present
  ✅ dev path present

Relevant config lines:
  train = "data/annotated/train.spacy"
  dev = "data/annotated/dev.spacy"
  vectors = null
  gpu_allocator = null
  batch_size = 128
  vectors = {"@vectors":"spacy.Vectors.v1"}
  include_static_vectors = true
  [corpora.dev]
  path = ${paths.dev}
  [corpora.train]
  path = ${paths.train}
  [training]
  dev_corpus = "corpora.dev"
  train_corpus = "corpora.train"
  gpu_allocator = ${system.gpu_allocator}
  [training.batcher]
  @batchers = "spacy.batch_by_words.v1"
  [training.batcher.size]
  [training.logger]
  [training.optimizer]
  [training.score_weights]
  [pretraining]
  vectors = ${paths.vectors}


In [3]:
# Notebook 03 — Cell 1: Updated config for RTX 5060 & 32GB RAM
with open('../config.cfg', 'r') as f:
    config = f.read()

# Increasing batch_size to utilize GPU VRAM (RTX 5060)
# Reverting patience and max_steps to higher values for better convergence
config = config.replace('batch_size = 32', 'batch_size = 128') 
config = config.replace('patience = 400',   'patience = 1200')
config = config.replace('max_steps = 2000', 'max_steps = 5000')

with open('../config.cfg', 'w') as f:
    f.write(config)

print("🚀 Config optimized for RTX 5060 & 32GB RAM")
for line in config.split('\n'):
    if any(x in line for x in ['batch_size','patience','max_steps']):
        print(f"  {line.strip()}")

🚀 Config optimized for RTX 5060 & 32GB RAM
  batch_size = 128
  patience = 1200
  max_steps = 5000


In [4]:
# Verify 2 — Data files exist and are the right size
import os

files = {
    '../data/annotated/train.spacy': 'should be largest',
    '../data/annotated/dev.spacy':   'should be ~20% of train',
    '../data/annotated/test.spacy':  'should be similar to dev',
    '../config.cfg':                 'training config',
}

print("File sizes:")
for path, note in files.items():
    size = os.path.getsize(path)/1024 if os.path.exists(path) else 0
    print(f"  {'✅' if size>0 else '❌'} {os.path.basename(path):<20} {size:>8.1f} KB  ({note})")

File sizes:
  ✅ train.spacy            4261.2 KB  (should be largest)
  ✅ dev.spacy               772.6 KB  (should be ~20% of train)
  ✅ test.spacy              859.4 KB  (should be similar to dev)
  ✅ config.cfg                2.8 KB  (training config)


In [5]:
# Verify 3 — Old model cleared
import shutil, os

for folder in ['../model/model-best', '../model/model-last']:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"✅ Cleared {folder}")
    else:
        print(f"✅ Already clean: {folder}")

✅ Cleared ../model/model-best
✅ Cleared ../model/model-last


In [6]:
# Verify 4 — Quick data sanity check
import json
from collections import Counter

with open('../data/synthetic/documents.json') as f:
    docs = json.load(f)

# Check split distribution
splits = Counter(d['split'] for d in docs)
types  = Counter(d['doc_type'] for d in docs)
labels = Counter(
    lbl
    for d in docs
    for _, _, lbl in d['entities']
)

print(f"Total docs: {len(docs)}")
print(f"\nBy split:   {dict(splits)}")
print(f"\nBy type:")
for k, v in types.most_common():
    print(f"  {k:<25} {v}")
print(f"\nBy entity label:")
for k, v in labels.most_common():
    print(f"  {k:<20} {v}")


Total docs: 7000

By split:   {'train': 6000, 'test': 1000}

By type:
  KYC                       1001
  LOAN_APPLICATION          1000
  WIRE_TRANSFER             1000
  ACCOUNT_STATEMENT         1000
  COMPLAINT_LETTER          1000
  SAR_MEMO                  1000
  INTERNAL_NOTE             999

By entity label:
  PERSON               9999
  ORG                  9001
  ACCOUNT_NO           8000
  AMOUNT               7999
  EMAIL                7872
  PHONE                7000
  POSTAL_CODE          6952
  ADDRESS              6000
  SIN                  4000
  DOB                  2869
  CREDIT_CARD          2860
  TRANSIT_NO           1001
  SWIFT                1000


In [7]:
# Fix config.cfg paths
with open('../config.cfg', 'r') as f:
    config = f.read()

# Fix the paths section
config = config.replace(
    'train = null',
    'train = "data/annotated/train.spacy"'
)
config = config.replace(
    'dev = null',
    'dev = "data/annotated/dev.spacy"'
)

with open('../config.cfg', 'w') as f:
    f.write(config)

# Verify fix
print("Verifying fix:")
for line in config.split('\n'):
    if line.strip().startswith('train =') or line.strip().startswith('dev ='):
        print(f"  {line.strip()}")


Verifying fix:
  train = "data/annotated/train.spacy"
  dev = "data/annotated/dev.spacy"


# Notebook 03 : Model Training
Fine-tunes spaCy en_core_web_trf on 4,000 annotated banking documents
Expected time: ~30-60 min CPU | ~5-10 min GPU

In [8]:
import subprocess

# Generate config.cfg (run once)
# r = subprocess.run(['python','-m','spacy','init','config','../config.cfg',
#                     '--lang','en','--pipeline','ner','--optimize','accuracy'],
#                    capture_output=True, text=True)
# print(r.stdout or r.stderr)

In [9]:
# Read config, fix the vectors line, write back
config_path = '../config.cfg'

with open(config_path, 'r') as f:
    config = f.read()

# Replace en_core_web_lg reference with null
config = config.replace(
    'vectors = "en_core_web_lg"',
    'vectors = null'
)

with open(config_path, 'w') as f:
    f.write(config)

print("config.cfg fixed — vectors set to null")
print("Checking fix:")
for line in config.split('\n'):
    if 'vectors' in line:
        print(" ", line)

config.cfg fixed — vectors set to null
Checking fix:
  vectors = null
  vectors = {"@vectors":"spacy.Vectors.v1"}
  include_static_vectors = true
  vectors = ${paths.vectors}


In [10]:
# Quick verify — show all relevant config lines
with open('../config.cfg', 'r') as f:
    config = f.read()

keywords = ['batch_size', 'patience', 'max_steps', 
            'eval_frequency', 'dropout', 'train =', 'dev =']

print("Key training settings:")
for line in config.split('\n'):
    if any(kw in line for kw in keywords):
        print(f"  {line.strip()}")

Key training settings:
  train = "data/annotated/train.spacy"
  dev = "data/annotated/dev.spacy"
  batch_size = 128
  dropout = 0.1
  patience = 1200
  max_steps = 5000
  eval_frequency = 200


In [11]:
import spacy
print(f"📦 spaCy Version: {spacy.__version__}")

📦 spaCy Version: 3.7.5


In [12]:
import subprocess
import sys
import os

# 1. Define the CUDA path for your RTX 5070
cuda_path = r"C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v12.8"

# 2. Prepare the environment for the subprocess
env = os.environ.copy()
if os.path.exists(cuda_path):
    env["CUDA_PATH"] = cuda_path
    # Add CUDA 'bin' and 'libnvvp' to the PATH so the subprocess sees the DLLs
    env["PATH"] = f"{cuda_path}\\bin;{cuda_path}\\libnvvp;" + env["PATH"]
    print("✅ CUDA environment prepared for subprocess.")
else:
    print("⚠️ Warning: CUDA path not found. Subprocess may fail again.")

# 3. Launch the training with the 'env' parameter
process = subprocess.Popen(
    [
        sys.executable, '-m', 'spacy', 'train', '../config.cfg',
        '--output',      '../model',
        '--paths.train', '../data/annotated/train.spacy',
        '--paths.dev',   '../data/annotated/dev.spacy',
        '--gpu-id',      '0'  # Targeting your RTX 5070
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    env=env  # <--- THIS IS THE KEY FIX
)

print("🚀 Training started on GPU: 0...")

# Print each line as it arrives
for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
print(f"\n✅ Done — returncode: {process.returncode}")

✅ CUDA environment prepared for subprocess.
🚀 Training started on GPU: 0...
ℹ Saving to output directory: ..\model
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================
✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['tok2vec', 'ner']
ℹ Initial learn rate: 0.001
E    #       LOSS TOK2VEC  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  ------------  --------  ------  ------  ------  ------
  0       0          0.00     62.80    0.00    0.00    0.00    0.00
  0     200       1126.88   5547.41   68.35   68.65   68.05    0.68
  0     400        194.03   2117.83   89.30   89.69   88.92    0.89
  0     600         67.53    649.76   97.76   98.05   97.47    0.98
  0     800         56.71    283.93   98.82   98.98   98.65    0.99
  0    1000         61.82    219.93   99.38   99.43   99.32    0.99
  0    1200         36.57     82.66   99.49   99.50   99.49    0.99
  0    1400       

In [13]:
# r = subprocess.run([
#     'python','-m','spacy','train','../config.cfg',
#     '--output','../model',
#     '--paths.train','../data/annotated/train.spacy',
#     '--paths.dev','../data/annotated/dev.spacy',
#     '--gpu-id','-1'
# ], capture_output=False, text=True)

# # With this — works with both capture_output=True and False:
# if r.stdout:
#     print(r.stdout[-3000:])
# if r.returncode != 0 and r.stderr:
#     print('ERR:', r.stderr[-1000:])
# else:
#     print("✅ Training completed — returncode:", r.returncode)

In [20]:
import os, json

# Check both folders
for folder in ['../model/model-best', '../model/model-last']:
    if os.path.exists(folder):
        files = os.listdir(folder)
        size  = sum(
            os.path.getsize(os.path.join(folder, f))
            for f in files
        ) / (1024*1024)
        print(f"✅ {folder} — {len(files)} files, {size:.1f} MB")
        
        # Show performance from meta.json
        meta_path = os.path.join(folder, 'meta.json')
        if os.path.exists(meta_path):
            with open(meta_path) as f:
                meta = json.load(f)
            perf = meta.get('performance', {})
            print(f"   F1:        {perf.get('ents_f', 0):.3f}")
            print(f"   Precision: {perf.get('ents_p', 0):.3f}")
            print(f"   Recall:    {perf.get('ents_r', 0):.3f}")
    else:
        print(f"❌ {folder} — not found")

✅ ../model/model-best — 6 files, 0.1 MB
   F1:        1.000
   Precision: 1.000
   Recall:    1.000
✅ ../model/model-last — 6 files, 0.1 MB
   F1:        0.999
   Precision: 0.999
   Recall:    0.999


In [21]:
import os
print("model-best exists:", os.path.exists('../model/model-best'))
print("model-last exists:", os.path.exists('../model/model-last'))

model-best exists: True
model-last exists: True


In [22]:
# Sanity check

import warnings
warnings.filterwarnings("ignore", message="\[W095\]")

import spacy
nlp = spacy.load('../model/model-best')

test_text = '''
Full Name: Priya Sharma
SIN: 482-716-930  Phone: 416-555-0192
Email: priya.s@gmail.com
Account: 00152-004-7823941
Address: 47 Lakeshore Blvd W, Toronto, ON M6K 1C3
Amount: $42,500.00
'''
for ent in nlp(test_text).ents:
    print(f'{ent.label_:<20} {ent.text}')

PERSON               Priya Sharma
SIN                  482-716-930  Phone: 416-555-0192
EMAIL                priya.s@gmail.com
ACCOUNT_NO           00152-004-7823941
ADDRESS              47 Lakeshore Blvd W, Toronto, ON M6K 1C3
AMOUNT               $42,500.00


In [23]:
# How many epochs did it actually complete?
import json

with open('../model/model-best/meta.json') as f:
    meta = json.load(f)

print("Model name:   ", meta.get('name'))
print("Language:     ", meta.get('lang'))
print("Pipeline:     ", meta.get('pipeline'))
print("Labels:       ", meta.get('labels'))

Model name:    pipeline
Language:      en
Pipeline:      ['tok2vec', 'ner']
Labels:        {'tok2vec': [], 'ner': ['ACCOUNT_NO', 'ADDRESS', 'AMOUNT', 'CREDIT_CARD', 'DOB', 'EMAIL', 'ORG', 'PERSON', 'PHONE', 'POSTAL_CODE', 'SIN', 'SWIFT', 'TRANSIT_NO']}


In [24]:
import spacy

nlp = spacy.load('../model/model-best')

test_docs = [
    "" "hey spoke to david mcallister his number is 416-334-9821 "
    "email d.mcallister@hotmail.com account 00341-006-4421987 "
    "at BMO owes $3,200.00",

    "fatima al-hassan called re her td bank account "
    "reach her at 514-882-3301 or fatima.h@yahoo.ca "
    "address 221 rue sainte-catherine montreal qc h2x 1l4 "
    "disputed $890.00 on card 4711-2233-4455-6677",

    "mike torres 905-334-7821 flagged transaction $15,750.00 "
    "account 00891-003-7712334 sin 527-384-910 dob 1979-11-03 "
    "address 99 wellington st w toronto on m5k 1j3",

    "wire transfer approved for jennifer wu at rbc sending $24,500.00 "
    "from 00234-002-8812334 to 00891-006-4421198 swift royccat2 "
    "her email jen.wu@gmail.com phone 647-221-8834",

    "loan application from roberto sanchez sin 482-331-092 "
    "dob 1990-04-15 employer maple tech inc income $78,000.00 "
    "requested $45,000.00 contact 905-441-2281 roberto.s@gmail.com "
    "address 55 queen st e toronto on m5c 1r6""",
]

print(f"{'Label':<20} {'Text'}")
print("-" * 50)
for i, text in enumerate(test_docs):
    print(f"\n--- Doc {i+1} ---")
    doc = nlp(text)
    if doc.ents:
        for ent in doc.ents:
            print(f"  {ent.label_:<20} {ent.text}")
    else:
        print("  ⚠️  No entities detected")

Label                Text
--------------------------------------------------

--- Doc 1 ---
  PERSON               david mcallister
  SIN                  416-334-9821
  ACCOUNT_NO           00341-006-4421987
  ORG                  BMO
  AMOUNT               $3,200.00

--- Doc 2 ---
  ORG                  td bank
  PHONE                514-882-3301
  ADDRESS              221 rue sainte-catherine montreal qc h2x 1l4
  AMOUNT               $890.00
  CREDIT_CARD          4711-2233-4455-6677

--- Doc 3 ---
  PHONE                905-334-7821
  AMOUNT               $15,750.00
  ACCOUNT_NO           00891-003-7712334
  SIN                  527-384-910
  DOB                  1979-11-03
  PERSON               99 wellington

--- Doc 4 ---
  PERSON               jennifer wu
  ORG                  rbc
  AMOUNT               $24,500.00
  ACCOUNT_NO           00234-002-8812334
  ACCOUNT_NO           00891-006-4421198
  SWIFT                royccat2
  EMAIL                jen.wu@gmail.com
  PHONE   

## Done : Next: 04_evaluation.ipynb

Doc Type          Before          After
─────────────────────────────────────────────────────
KYC               1 template      5 headers
                                  16 name label variants
                                  shuffled field order

Wire Transfer     1 template      4 headers
                                  4 narrative variants
                                  shuffled sender/receiver

Loan Application  1 template      4 headers
                                  4 officer note variants
                                  shuffled field order

Account Statement 1 template      4 headers
                                  shuffled fields

SAR Memo          1 template      4 headers
                                  4 narrative variants

Complaint Letter  1 template      4 headers
                                  4 body styles

Internal Note     4 templates  →  5 templates (added 5th)

In [25]:
import spacy

nlp_base = spacy.load('en_core_web_trf')

tests = [
    "fatima al-hassan called re her td bank account at 514-882-3301",
    "mike torres at bmo flagged transaction of $15,750.00",
    "david mcallister emailed d.mcallister@hotmail.com from royal bank",
    "roberto sanchez applied for loan at cibc income $78,000.00",
]

print("BASE MODEL — en_core_web_trf")
print("-" * 50)
for i, text in enumerate(tests):
    print(f"\n--- Doc {i+1} ---")
    doc = nlp_base(text)
    if doc.ents:
        for ent in doc.ents:
            print(f"  {ent.label_:<20} {ent.text}")
    else:
        print("  No entities detected")

BASE MODEL — en_core_web_trf
--------------------------------------------------

--- Doc 1 ---
  PERSON               fatima al-
  ORG                  td

--- Doc 2 ---
  PERSON               mike torres
  ORG                  bmo
  MONEY                15,750.00

--- Doc 3 ---
  PERSON               david mcallister
  ORG                  royal bank

--- Doc 4 ---
  PERSON               roberto sanchez
  ORG                  cibc
  MONEY                78,000.00


c:\Users\manpr\anaconda3\envs\ner-pipeline-2\Lib\site-packages\thinc\shims\pytorch.py:114: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self._mixed_precision):
